# Pandas met Fred
We gaan macro-economische data inlezen bij de FED in St. Louis: https://fred.stlouisfed.org/categories

In [1]:
! pip install pandas-datareader==0.10.0
! pip install yfinance

import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import date

#Clear output
from google.colab import output
output.clear()

Alle libraries geinstalleerd, nu de data:

In [3]:
yf.pdr_override()

start = dt.datetime(2017,11,15)
end = dt.datetime(2022,12,15)

In [4]:
inflatie = web.get_data_fred('T10YIE', start, end) #inflatie
rente = web.get_data_fred('FEDFUNDS', start, end) #rente
graan = web.get_data_fred('PWHEAMTUSDM', start, end) #graan
koper = web.get_data_fred('PCOPPUSDM', start, end) #koperprijs
koffie = web.get_data_fred('WPU026301', start, end) #koffiebonen
hout = web.get_data_fred('WPU08', start, end) #hout
vlees = web.get_data_fred('PBEEFUSDM', start, end) #vleesprijs
zonnebloem = web.get_data_fred('PSUNOUSDM', start, end) #zonnebloemolie
wol = web.get_data_fred('PWOOLFUSDM', start, end) #wol
katoen = web.get_data_fred('PCOTTINDUSDM', start, end) #katoen
rubber = web.get_data_fred('PRUBBUSDM', start, end) #rubber
oil = web.get_data_yahoo('CL=F', start, end).tz_localize(None) #olie 
goud = web.get_data_yahoo('GC=F', start, end).tz_localize(None) #goud

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Even kijken hoe het eruit ziet:

In [5]:
hout.head()

,WPU08
DATE,
2017-12-01,235.6
2018-01-01,237.3
2018-02-01,242.4
2018-03-01,244.7
2018-04-01,244.8


Of gaan we straks de regressie doen op de prijsveranderingen per dag:

In [ ]:
hout.pct_change()

,WPU08
DATE,
2017-12-01,NaN
2018-01-01,0.007216
2018-02-01,0.021492
2018-03-01,0.009488
2018-04-01,0.000409
2018-05-01,0.020016
2018-06-01,0.013216
2018-07-01,-0.008696
2018-08-01,-0.018341


Sowieso toe te voegen aan deze tabellen: is dit een prijs per dag? In welke eenheid? Per m3 of liter of vat of ..

Afijn: eerst maar weer de portfolio-data inlezen:

In [ ]:
apple = web.get_data_yahoo('AAPL', start, end).tz_localize(None)
btc = web.get_data_yahoo('BTC-USD', start, end).tz_localize(None)
eth = web.get_data_yahoo('ETH-USD', start, end).tz_localize(None)
bond = web.get_data_yahoo('^TYX', start, end).tz_localize(None)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
portfolio = pd.concat([apple, btc, eth, bond], axis=1)
portfolio = pd.concat([apple.add_suffix("_Apple"), btc.add_suffix("_Bitcoin"), eth.add_suffix("_Ethereum"),bond.add_suffix("_Bond")], axis=1)
portfolio_gefilterd = portfolio.loc[:, ["Close_Bitcoin", "Close_Ethereum", "Close_Bond", "Close_Apple"]]
nans_gevuld = portfolio_gefilterd.ffill()
nans_gevuld

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple
Date,,,,
2017-11-15,7315.540039,333.356995,2.781,42.270000
2017-11-16,7871.689941,330.924011,2.806,42.775002
2017-11-17,7708.990234,332.394012,2.793,42.537498
2017-11-18,7790.149902,347.612000,2.793,42.537498
2017-11-19,8036.490234,354.385986,2.793,42.537498
...,...,...,...,...
2022-12-10,17128.724609,1266.384155,3.549,142.160004
2022-12-11,17104.193359,1263.868530,3.549,142.160004
2022-12-12,17206.437500,1274.619019,3.576,144.490005


En nu het 'piece-de-resistance': beide tabellen samenvoegen:

In [ ]:
apt = pd.concat([nans_gevuld, hout], axis=1)
apt_fill = apt.ffill()
apt_fill.head(10)

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-11-15,7315.540039,333.356995,2.781,42.270000,NaN
2017-11-16,7871.689941,330.924011,2.806,42.775002,NaN
2017-11-17,7708.990234,332.394012,2.793,42.537498,NaN
2017-11-18,7790.149902,347.612000,2.793,42.537498,NaN
2017-11-19,8036.490234,354.385986,2.793,42.537498,NaN
2017-11-20,8200.639648,366.730011,2.793,42.494999,NaN
2017-11-21,8071.259766,360.401001,2.763,43.285000,NaN
2017-11-22,8253.549805,380.652008,2.746,43.740002,NaN
2017-11-23,8038.770020,410.165985,2.746,43.740002,NaN
2017-11-24,8253.690430,474.911011,2.762,43.742500,NaN


Nu alles omzetten naar prijsveranderingen:

In [ ]:
return_portfolio = apt_fill.pct_change()
return_portfolio.head(10)

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-11-15,NaN,NaN,NaN,NaN,NaN
2017-11-16,0.076023,-0.007298,0.008990,0.011947,NaN
2017-11-17,-0.020669,0.004442,-0.004633,-0.005552,NaN
2017-11-18,0.010528,0.045783,0.000000,0.000000,NaN
2017-11-19,0.031622,0.019487,0.000000,0.000000,NaN
2017-11-20,0.020426,0.034832,0.000000,-0.000999,NaN
2017-11-21,-0.015777,-0.017258,-0.010741,0.018590,NaN
2017-11-22,0.022585,0.056190,-0.006153,0.010512,NaN
2017-11-23,-0.026023,0.077535,0.000000,0.000000,NaN
2017-11-24,0.026735,0.157851,0.005827,0.000057,NaN


We zijn nu bijna waar we wezen willen: alle rijen en kolommen gevuld. Laatste schoonmaakactie: het verwijderen / filteren van alle overgebleven NaN's.<p>
Dit doen we door als startdatum een datum te kiezen waarbij voor ALLE producten koersdata aanwezig zijn.
We zoeken eerst deze datum:

In [ ]:
portfolio_regressie = return_portfolio.loc[return_portfolio.loc[:, "Close_Ethereum"] < 0, :]
portfolio_regressie

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-11-16,0.076023,-0.007298,0.008990,0.011947,NaN
2017-11-21,-0.015777,-0.017258,-0.010741,0.018590,NaN
2017-11-25,0.065090,-0.018182,0.000000,0.000000,NaN
2017-11-28,0.024490,-0.015516,0.000362,-0.005859,NaN
2017-11-29,-0.016919,-0.095959,0.018438,-0.020743,NaN
...,...,...,...,...,...
2022-12-05,-0.009087,-0.016075,0.015160,-0.007983,0.0
2022-12-07,-0.014124,-0.030839,-0.030656,-0.013785,0.0
2022-12-09,-0.005821,-0.013138,0.027207,-0.003435,0.0
2022-12-11,-0.001432,-0.001986,0.000000,0.000000,0.0


De datum die we zoeken is 17 november 2017. Deze datum kunnen we nu gebruiken als startdatum voor onze portfolio.

# Hoe kun je nog meer data samenvoegen?
Via een join, een append of een merge:

In [ ]:
nans_gevuld.join(hout)

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
Date,,,,,
2017-11-15,7315.540039,333.356995,2.781,42.270000,NaN
2017-11-16,7871.689941,330.924011,2.806,42.775002,NaN
2017-11-17,7708.990234,332.394012,2.793,42.537498,NaN
2017-11-18,7790.149902,347.612000,2.793,42.537498,NaN
2017-11-19,8036.490234,354.385986,2.793,42.537498,NaN
...,...,...,...,...,...
2022-12-10,17128.724609,1266.384155,3.549,142.160004,NaN
2022-12-11,17104.193359,1263.868530,3.549,142.160004,NaN
2022-12-12,17206.437500,1274.619019,3.576,144.490005,NaN


In [ ]:
nans_gevuld.append(hout)

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-11-15,7315.540039,333.356995,2.781,42.270000,NaN
2017-11-16,7871.689941,330.924011,2.806,42.775002,NaN
2017-11-17,7708.990234,332.394012,2.793,42.537498,NaN
2017-11-18,7790.149902,347.612000,2.793,42.537498,NaN
2017-11-19,8036.490234,354.385986,2.793,42.537498,NaN
...,...,...,...,...,...
2022-07-01,NaN,NaN,NaN,NaN,331.669
2022-08-01,NaN,NaN,NaN,NaN,329.173
2022-09-01,NaN,NaN,NaN,NaN,322.706
2022-10-01,NaN,NaN,NaN,NaN,316.694


De append voegt alleen de maandelijkse houtprijzen toe, maar combineert deze niet met de bestaande data. Voor ons doel dus  niet bruikbaar.

In [ ]:
apt = pd.merge(nans_gevuld, hout, left_index=True, right_on='DATE')
apt = pd.merge(nans_gevuld, hout, left_index=True, right_index=True)
apt

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-12-01,10975.599609,466.540009,2.758,42.762501,235.600
2018-01-01,13657.200195,772.640991,2.739,42.307499,237.300
2018-02-01,9170.540039,1036.790039,3.005,41.945000,242.400
2018-03-01,10951.000000,872.200012,3.088,43.750000,244.700
2018-04-01,6844.229980,379.606995,2.972,41.945000,244.800
2018-05-01,9119.009766,673.612976,3.137,42.275002,249.700
2018-06-01,7541.450195,580.043030,3.046,47.560001,253.000
2018-07-01,6385.819824,453.917999,2.983,46.277500,250.800
2018-08-01,7624.910156,420.747009,3.127,50.375000,246.200
2018-09-01,7193.250000,295.341003,3.010,56.907501,244.300


Keurig alles gevuld ..  maar we houden maar 116 rijen over? Rara hoe kan dit?
Omdat de hout-tabel maar 130 rijen heeft!

In [ ]:
apt = pd.merge(nans_gevuld, hout, how='outer', left_index=True, right_index=True)
apt

,Close_Bitcoin,Close_Ethereum,Close_Bond,Close_Apple,WPU08
2017-11-15,7315.540039,333.356995,2.781,42.270000,NaN
2017-11-16,7871.689941,330.924011,2.806,42.775002,NaN
2017-11-17,7708.990234,332.394012,2.793,42.537498,NaN
2017-11-18,7790.149902,347.612000,2.793,42.537498,NaN
2017-11-19,8036.490234,354.385986,2.793,42.537498,NaN
...,...,...,...,...,...
2022-12-10,17128.724609,1266.384155,3.549,142.160004,NaN
2022-12-11,17104.193359,1263.868530,3.549,142.160004,NaN
2022-12-12,17206.437500,1274.619019,3.576,144.490005,NaN
2022-12-13,17781.318359,1320.549194,3.528,145.470001,NaN


Ergo: als we gaan samenvoegen, moeten we eerst de ontbrekende waarden in de kolommen aanvullen met forward_fill. <br>
Kunnen we maandelijkse / jaarlijkse data op dagbasis aanvullen? Ja: zie hierboven.